# Make Predictions

In [ ]:
import os
import sys
sys.path.append('../')
import time
import torch
import skimage
import sklearn.metrics
import wandb

import numpy as np
import matplotlib.pyplot as plt

import mnds
import mnmodel
import evaluation

# Hyperparameter setting

In [ ]:
CURRENT_PATH = os.getcwd()
DIRECTORY = CURRENT_PATH + '/all_data_micronuclei/'
OUTPUT_DIR = "/model_output/output/"

# set CHTC writeable cache directory for pytorch and matplotlib
os.environ['TORCH_HOME'] = CURRENT_PATH + '/.cache/torch'
os.environ['MPLCONFIGDIR'] = CURRENT_PATH + '/.cache/matplotlib/config'
torch.set_num_threads(8)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SCALE_FACTOR = 1.0
PATCH_SIZE = 256
STRIDE = 8
FEATURE_SIZE = 384
TOKENS_PER_PATCH = PATCH_SIZE // STRIDE
STEP = 16
EPOCHS = 20
THRESHOLD = 0.5
ANNOTATION_TYPE = 'edge' # train on our own data

LOSS_FN = 'combined'
LR = 1e-5
BATCH_SIZE = 32
FINETUNE = True
WEIGHT_DECAY = 1e-6

gpu = 0
device = f"cuda:{gpu}" if torch.cuda.is_available() else 'cpu'

# Load Model

In [ ]:
files = os.listdir(DIRECTORY)
filelist = [file for file in files if not file.startswith('.')]
annot_files = [x for x in filelist if x.endswith('png')]
annot_files.sort()

predictions_dir = DIRECTORY + OUTPUT_DIR
models_dir = OUTPUT_DIR

model = mnmodel.MicronucleiModel(DIRECTORY, device, patch_size=PATCH_SIZE, edges=True)
model.load('best_model_v3.pth', model_dir=models_dir)

# Make Predictions

In [ ]:
validation_file = 'C2-20X_c0-DAPI-GFP_B3_Tile-16.phenotype_outlines.png'
imid = validation_file.split('.')[0]

im = mnds.read_image(DIRECTORY, imid, 'phenotype.tif', scale=SCALE_FACTOR)
im = np.array((im - np.min(im))/(np.max(im) - np.min(im)), dtype="float32")

probabilities = model.predict(im, stride=1, step=STEP, batch_size=BATCH_SIZE)
filename = predictions_dir + validation_file.replace('phenotype_outlines.png','_probabilities')

# Save Predictions

In [ ]:
mn_pred = mn_pred = probabilities[0,:,:] > THRESHOLD
np.save(filename, mn_pred)

In [ ]:
input = skimage.io.imread(f'{os.getcwd()}/dataset_v2/{imid}.phenotype.tif')

fig, ax = plt.subplots(1,3,figsize=(10,6))
for i in range(2):
    ax[i].get_xaxis().set_visible(False)
    ax[i].get_yaxis().set_visible(False)

ax[0].imshow(input)
ax[0].set_title('Original Input')
ax[1].imshow(mn_pred)
ax[1].set_title('Our Model Prediction')